# PiShield — a Shield Layer for linear requirements

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihaela-stoian/PiShield/blob/hierarchical-requirements/examples/shield_layer_linear.ipynb)

This notebook runs **end-to-end with no external downloads**. We train a tiny MLP whose
3-dimensional output must be **non-increasing** (`y_0 >= y_1 >= y_2`) — a set of **linear**
requirements. By wrapping the model with a Shield Layer, its outputs satisfy that requirement
*at every step* — during training and at inference. We compare against an unconstrained baseline
to see the difference.

## Setup

Uses a local PiShield checkout if you're running inside one; otherwise installs PiShield
(e.g. on a fresh Colab runtime).

In [ ]:
import importlib.util, subprocess, sys, os

root = os.path.abspath(os.getcwd())
while root != os.path.dirname(root) and not os.path.isdir(os.path.join(root, 'pishield')):
    root = os.path.dirname(root)
if os.path.isdir(os.path.join(root, 'pishield')):
    sys.path.insert(0, root)
    print('Using local PiShield at', root)
elif importlib.util.find_spec('pishield') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'git+https://github.com/mihaela-stoian/PiShield.git@hierarchical-requirements'],
                   check=True)
    print('Installed PiShield')
else:
    print('PiShield already available')

In [ ]:
import torch
import torch.nn as nn
from tqdm.auto import tqdm

from pishield.shield_layer import build_shield_layer

torch.manual_seed(0)

## 1. A tiny synthetic task

Random inputs, and non-increasing 3-vector targets. We use **small gaps** between consecutive
values, so the ordering is easy to get *wrong*: an unconstrained model will tend to invert
some of them, which is exactly what the requirement forbids.

In [ ]:
N, in_dim, out_dim = 512, 4, 3

X = torch.randn(N, in_dim)
base = torch.rand(N, 1)
gaps = torch.rand(N, 2) * 0.05  # small gaps -> ordering is non-trivial to preserve
Y = torch.cat([base, base - gaps[:, :1], base - gaps.sum(1, keepdim=True)], dim=1)
print("example target (non-increasing):", [round(v, 2) for v in Y[0].tolist()])

## 2. Write the requirements file (`y_0 >= y_1 >= y_2`)

In [ ]:
requirements_path = "monotonic_requirements.txt"
with open(requirements_path, "w") as f:
    f.write("ordering y_0 y_1 y_2\ny_0 - y_1 >= 0\ny_1 - y_2 >= 0\n")

## 3. Define the models

`ShieldedMLP` simply applies the Shield Layer to the MLP's output inside `forward`; that is
the *only* change compared to the plain baseline.

In [ ]:
def make_mlp():
    return nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU(), nn.Linear(32, out_dim))

class ShieldedMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = make_mlp()
        self.shield = build_shield_layer(out_dim, requirements_path)

    def forward(self, x):
        return self.shield(self.model(x))

## 4. Train both models and compare

We count how many outputs violate the requirement. The same training loop works for both,
because the Shield Layer is differentiable and gradients flow through it.

In [ ]:
def num_violations(out, tol=1e-6):
    bad = (out[:, 0] < out[:, 1] - tol) | (out[:, 1] < out[:, 2] - tol)
    return int(bad.sum())

def train(model, label, epochs=300, lr=1e-2):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    progress = tqdm(range(epochs), desc=f'training {label}')
    for _ in progress:
        opt.zero_grad()
        loss = loss_fn(model(X), Y)
        loss.backward()
        opt.step()
        progress.set_postfix(loss=f'{loss.item():.4f}')
    return loss.item()

baseline = make_mlp()
shielded = ShieldedMLP()

base_mse = train(baseline, label='unconstrained NN')
shield_mse = train(shielded, label='NN + PiShield')

with torch.no_grad():
    base_out, shield_out = baseline(X), shielded(X)

print(f"baseline : final MSE={base_mse:.4f}, violations={num_violations(base_out)}/{N}")
print(f"shielded : final MSE={shield_mse:.4f}, violations={num_violations(shield_out)}/{N}")

## 5. Visualising the constraints

Each requirement is a half-plane. In the grid below, the columns are the two constraints (the
shaded area is infeasible and the dashed line is the boundary) and the rows are the
**unconstrained NN**, **NN + PiShield**, and the **real data** (the training targets). The
unconstrained NN scatters points into the infeasible region — the violations counted above —
while NN + PiShield stays on the feasible side, together with the real data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

base = base_out.numpy()
shield = shield_out.numpy()
real = Y.numpy()

# (x-column, y-column, x-label, y-label, constraint title)
constraints = [(1, 0, '$y_1$', '$y_0$', '$y_0 - y_1 \\geq 0$'),
               (2, 1, '$y_2$', '$y_1$', '$y_1 - y_2 \\geq 0$')]
rows = [('unconstrained NN', base, '#3f72c4'),
        ('NN + PiShield', shield, '#5a9e4a'),
        ('real data', real, '#8a8a8a')]

allv = np.concatenate([base, shield, real])
lo, hi = float(allv.min()) - 0.05, float(allv.max()) + 0.05

fig, axes = plt.subplots(3, 2, figsize=(8.6, 12.5))
for r, (name, out, color) in enumerate(rows):
    for c, (xi, yi, xl, yl, ctitle) in enumerate(constraints):
        ax = axes[r, c]
        ax.fill([lo, hi, hi], [lo, lo, hi], color='#e7a6a6', alpha=0.4, lw=0, zorder=0)  # infeasible region
        ax.plot([lo, hi], [lo, hi], '--', color='#9a9a9a', lw=1, zorder=1)                # constraint boundary
        ax.scatter(out[:, xi], out[:, yi], s=30, c=color, alpha=0.6, edgecolors='none', zorder=2)
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect('equal')
        ax.set_xlabel(xl); ax.set_ylabel(yl); ax.grid(True, ls=':', alpha=0.3)
        if r == 0:
            ax.set_title(ctitle, fontsize=13)
    axes[r, 0].annotate(name, xy=(-0.28, 0.5), xycoords='axes fraction', rotation=90,
                        va='center', ha='center', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Takeaway

The **shielded** model has **zero** violations by construction — the Shield Layer enforces
the requirement on every forward pass — while still learning the task at least as well
(here it even reaches a slightly lower MSE). The unconstrained baseline minimises the MSE too, but leaves many outputs violating
the requirement.